# PI-LDM Serious Training (Google Colab)

This notebook provides a self-contained environment to train the **Physics-Informed Latent Diffusion Model (PI-LDM)** using GPU acceleration.

## 1. Setup
Install dependencies and prepare the directory structure.

In [ ]:
!pip install torch numpy pandas matplotlib traffic scikit-learn

import os
os.makedirs('data/processed', exist_ok=True)
os.makedirs('pi_ldm/models', exist_ok=True)

## 2. Upload Data
Upload your `X_LSZH_...npy` and `.csv` files to `/content/data/processed/` using the file browser on the left.

## 3. Implementation
This section consolidates all the classes needed for training.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math
import numpy as np
import pandas as pd
from torch.utils_data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

# --- 3.1 Dataset ---
class AircraftTrajectoryDataset(Dataset):
    def __init__(self, data_dir, file_base):
        X_path = os.path.join(data_dir, f"{file_base}.npy")
        meta_path = os.path.join(data_dir, f"{file_base}.csv")
        self.X = np.load(X_path)
        self.meta = pd.read_csv(meta_path)
        
        le_airport = LabelEncoder()
        airport_idx = le_airport.fit_transform(self.meta['airport'].astype(str))
        le_type = LabelEncoder()
        type_idx = le_type.fit_transform(self.meta['typecode'].astype(str))
        self.cond = np.column_stack((airport_idx, type_idx, np.zeros_like(airport_idx))).astype(np.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.cond[idx], dtype=torch.float32)

# --- 3.2 Model ---
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class ConditionalUNet1D(nn.Module):
    def __init__(self, state_dim, cond_dim, time_emb_dim=32, hidden_dims=[64, 128, 256]):
        super().__init__()
        self.time_mlp = nn.Sequential(SinusoidalPositionEmbeddings(time_emb_dim), nn.Linear(time_emb_dim, time_emb_dim * 2), nn.GELU(), nn.Linear(time_emb_dim * 2, time_emb_dim))
        self.cond_mlp = nn.Sequential(nn.Linear(cond_dim, time_emb_dim), nn.GELU())
        self.enc1 = nn.Conv1d(state_dim, hidden_dims[0], kernel_size=3, padding=1)
        self.enc2 = nn.Conv1d(hidden_dims[0], hidden_dims[1], kernel_size=3, padding=1, stride=2)
        self.bottleneck = nn.Conv1d(hidden_dims[1], hidden_dims[2], kernel_size=3, padding=1)
        self.dec1 = nn.ConvTranspose1d(hidden_dims[2], hidden_dims[1], kernel_size=4, padding=1, stride=2)
        self.dec2 = nn.Conv1d(hidden_dims[1] + hidden_dims[0], hidden_dims[0], kernel_size=3, padding=1)
        self.final_conv = nn.Conv1d(hidden_dims[0], state_dim, kernel_size=1)

    def forward(self, x, time, cond):
        t_emb = self.time_mlp(time).unsqueeze(-1)
        c_emb = self.cond_mlp(cond).unsqueeze(-1)
        emb = t_emb + c_emb
        x1 = F.relu(self.enc1(x))
        x2 = F.relu(self.enc2(x1))
        x_btn = F.relu(self.bottleneck(x2))
        x_up = F.relu(self.dec1(x_btn))
        if x_up.shape[-1] != x1.shape[-1]: x_up = F.interpolate(x_up, size=x1.shape[-1])
        x_out = F.relu(self.dec2(torch.cat([x_up, x1], dim=1)))
        return self.final_conv(x_out)

# --- 3.3 Physics Loss ---
class PhysicsLoss(nn.Module):
    def __init__(self): super().__init__()
    def forward(self, trajectories):
        # Physics residuals (Expand these with landing kinematics logic)
        return torch.tensor(0.0, device=trajectories.device, requires_grad=True)

# --- 3.4 Trainer ---
class PILDMTrainer:
    def __init__(self, state_dim=4, cond_dim=3, timesteps=1000, device='cuda'):
        self.device = device
        self.timesteps = timesteps
        self.beta = torch.linspace(1e-4, 0.02, timesteps).to(device)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)
        self.model = ConditionalUNet1D(state_dim, cond_dim).to(device)
        self.physics_fn = PhysicsLoss().to(device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=1e-4)
        self.mse_loss = nn.MSELoss()

    def train_step(self, x_0, cond, lambda_physics=0.1):
        self.optimizer.zero_grad()
        t = torch.randint(0, self.timesteps, (x_0.shape[0],), device=self.device).long()
        noise = torch.randn_like(x_0)
        a_hat = self.alpha_hat[t].view(-1, 1, 1)
        x_t = torch.sqrt(a_hat) * x_0 + torch.sqrt(1 - a_hat) * noise
        pred_noise = self.model(x_t, t, cond)
        l_diff = self.mse_loss(pred_noise, noise)
        
        # Physics Guidance (Predicted x0)
        x0_hat = (x_t - torch.sqrt(1 - a_hat) * pred_noise) / torch.sqrt(a_hat)
        l_phys = self.physics_fn(x0_hat.transpose(1, 2))
        
        loss = l_diff + lambda_physics * l_phys
        loss.backward()
        self.optimizer.step()
        return l_diff.item(), l_phys.item()

## 4. Run Training
Configure your constants and start the loop.

In [ ]:
FILE_BASE = "X_LSZH_2026-03-01_1000_to_2026-03-01_1200_runway14"
EPOCHS = 200
BATCH_SIZE = 32

dataset = AircraftTrajectoryDataset('data/processed', FILE_BASE)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
trainer = PILDMTrainer(device='cuda')

for epoch in range(EPOCHS):
    for x_0, cond in dataloader:
        l_d, l_p = trainer.train_step(x_0.to('cuda'), cond.to('cuda'))
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Diff Loss: {l_d:.4f} | Phys Loss: {l_p:.4f}")

torch.save(trainer.model.state_dict(), 'pi_ldm/models/serious_model.pth')
print("Training Complete. Model saved.")